# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a walk-through for loading and exploring the FAIR² Croissant dataset using the `mlcroissant` library. All dataset elements (record sets, fields, and columns) are referenced by their unique `@id`.

### Dataset Source
The dataset source Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

<br>

In [ ]:
# Install mlcroissant if it's not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a Python object
metadata = dataset.metadata

# Print key metadata fields
print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's inspect the available record sets and fields using their `@id`.

In [ ]:
# The following code finds all available record sets defined by their @id.
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
print(f"Available Record Sets (@id):\n{json.dumps(record_set_ids, indent=2)}\n")

if not record_set_ids:
    print("No explicit record sets listed in Croissant 'recordSet'. Attempting to auto-discover from underlying resources...")

# Show full field and column @id per record set
for record_set in record_set_ids:
    print(f"RecordSet @id: {record_set}")
    # Get fields for this record set
    recset_obj = None
    for rs in dataset.to_json()['recordSet']:
        if rs['@id'] == record_set:
            recset_obj = rs
            break
    fields = recset_obj.get('field', []) if recset_obj else []
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"  Field @id: {field_id}")
    print()

## Explore available records
List a sample of records from the **main record set** (replace `<id_of_the_records_set>` with the correct @id as found above).

In [ ]:
# Try to infer some available record sets by inspecting dataset.records() generators
# Here we attempt to discover the available record set @ids for extraction

# If explicit record sets are empty, check the `distribution` for data resources
main_record_set_id = None
if not record_set_ids:
    dist = dataset.metadata.to_json().get('distribution', [])
    if dist:
        # Try to guess the main resource
        main_record_set_id = dist[0]['@id']
    else:
        main_record_set_id = None
else:
    main_record_set_id = record_set_ids[0]

print(f"Using record set: {main_record_set_id}\n")

# List records
if main_record_set_id:
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record set id could be determined.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

All entities (record set, fields, columns) are referenced by their `@id`.

In [ ]:
# We'll try to extract data from all available record sets.
record_sets_to_extract = [main_record_set_id] if main_record_set_id else []
dataframes = {}

for rs_id in record_sets_to_extract:
    print(f"Loading records from record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns (@id): {df.columns.tolist()}")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

We will use a numeric field referenced by its `@id` (update as per the extracted DataFrame columns list).

In [ ]:
# Example: Suppose the main numeric field is named 'coverage_percent' (replace with correct column @id as needed)
df = dataframes[main_record_set_id]

# Find a numeric field for demonstration
numeric_columns = df.select_dtypes('number').columns.tolist()
print(f"Numeric fields found: {numeric_columns}")

if numeric_columns:
    numeric_field_id = numeric_columns[0]  # use the first numeric field for analysis
    print(f"Using numeric field for EDA: {numeric_field_id}\n")

    threshold = df[numeric_field_id].mean()  # arbitrary threshold for demonstration
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a string/categorical field
    group_fields = df.select_dtypes('object').columns.tolist()
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric fields found in this record set.")

## 5. Visualization
Visualize data distributions or field relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If there's a group field, plot group means
    if group_fields:
        group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values()
        plt.figure(figsize=(10, 4))
        group_means.plot(kind='bar')
        plt.title(f"Average {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()


## 6. Conclusion
This notebook demonstrated how to load, explore, and visualize the FAIR² Croissant dataset using the `mlcroissant` library. We:
- Programmatically discovered record sets and field `@id`s.
- Loaded records into DataFrames using only `@id` references.
- Performed basic EDA, normalization, filtering, and grouping using those IDs.
- Visualized distributions and group means.

Remember, all manipulations are done using the unique identifiers provided by the Croissant schema for full FAIR compliance!